# Figure 4: mRNA compositional relationship arises from transcription-translation balance

In [2]:
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '../analysis')
from model1 import run_model, plot_log2fc_heatmap
from bayesian_model1_ecoli import fit_model, predict_from_trace, \
    plot_protein_fractions, plot_mrna_fractions
from load_ecoli_fractions import label_genes, compute_fractions, REMOVE_MRNA, REMOVE_PROTEIN
from load_spombe_fractions import load_annotations, compute_mrna_fractions, compute_protein_fractions
from bayesian_model1_spombe import (fit_model_spombe, predict_from_trace_spombe,
                                     plot_protein_fractions_spombe, plot_mrna_fractions_spombe)

import bokeh.io
import bokeh.plotting
bokeh.io.output_notebook()

import arviz as az


Loading BokehJS ...

## Figure 4B: Hierarchical clustering remake

In [3]:
results = run_model()
heatmap = plot_log2fc_heatmap(results)
heatmap

:HeatMap   [Growth rate,mRNA species]   (log2FC)

## Figure 4C: Model 1 fit to E. coli data

**Load mRNA, protein, and growth data**

In [4]:
df_mrna_data   = pd.read_excel('../data/e.coli/science.abk2066_table_s3.xlsx', sheet_name=1)
df_mrna_growth = pd.read_excel('../data/e.coli/science.abk2066_table_s3.xlsx', sheet_name=0)

df_prot_data   = pd.read_excel('../data/e.coli/science.abk2066_table_s4.xlsx', sheet_name=1)
df_prot_growth = pd.read_excel('../data/e.coli/science.abk2066_table_s4.xlsx', sheet_name=0)

**Load E. coli annotation file**

In [5]:
df_ann = pd.read_csv('../data/e.coli/Schmidt2016_proteomics.csv')

**Define mRNA and protein fractions**

In [6]:
gene_to_label = label_genes(df_ann)


mrna_fracs = compute_fractions(df_mrna_data, df_mrna_growth, gene_to_label,
                               remove_samples=REMOVE_MRNA)


prot_fracs = compute_fractions(df_prot_data, df_prot_growth, gene_to_label,
                               remove_samples=REMOVE_PROTEIN)

# Define arrays for Bayesian fitting
mrna_growth = mrna_fracs['Growth rate'].values
mrna_r_frac = mrna_fracs['R'].values
mrna_c_frac = mrna_fracs['C'].values
mrna_q_frac = mrna_fracs['O'].values

protein_growth = prot_fracs['Growth rate'].values
R_P_frac = prot_fracs['R'].values
C_P_frac = prot_fracs['C'].values
Q_P_frac = prot_fracs['O'].values

**Fit to bayesian model and output plots with data overlayed**

In [7]:
trace = fit_model(protein_growth, R_P_frac, Q_P_frac, C_P_frac,
                  mrna_growth, mrna_r_frac, mrna_c_frac, mrna_q_frac)

preds  = predict_from_trace(trace, protein_growth, mrna_growth)
p_prot = plot_protein_fractions(protein_growth, R_P_frac, C_P_frac, Q_P_frac, preds)
p_mrna = plot_mrna_fractions(mrna_growth, mrna_r_frac, mrna_c_frac, mrna_q_frac, preds)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [gamma_1, alpha_q0, sigma_d, d_mrna_r, d_mrna_c, d_mrna_q, sigma_protein, sigma_mrna, dmrna/gamma_2]


Output()

Sampling 4 chains for 3_000 tune and 3_000 draw iterations (12_000 + 12_000 draws total) took 7 seconds.


In [8]:
az.summary(trace)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
gamma_1,2.365,0.089,2.207,2.530,0.001,0.001,8763.0,7065.0,1.0
alpha_q0,0.451,0.010,0.433,0.470,0.000,0.000,9143.0,7270.0,1.0
sigma_d,0.334,0.133,0.130,0.592,0.002,0.002,6462.0,7941.0,1.0
d_mrna_r,0.804,0.172,0.487,1.122,0.003,0.003,4064.0,4053.0,1.0
d_mrna_c,1.432,0.318,0.896,2.061,0.005,0.006,4125.0,4561.0,1.0
d_mrna_q,0.932,0.199,0.566,1.308,0.003,0.004,3996.0,4021.0,1.0
sigma_protein,0.046,0.005,0.037,0.057,0.000,0.000,8444.0,7368.0,1.0
sigma_mrna,0.057,0.006,0.045,0.069,0.000,0.000,8185.0,7061.0,1.0
dmrna/gamma_2,4.546,2.396,1.126,9.011,0.026,0.037,9266.0,7824.0,1.0


## Figure 4D: Model 1 fit to S. pombe data

**Load annotation files**

In [10]:
# Load annotations
ribo_ids, mito_ids = load_annotations(
    '../analysis/s.pombe_annotations/structuralConstituentofRibosome_pombase.tsv',
    '../analysis/s.pombe_annotations/mitophagy_pombase.tsv'
)


**Load mRNA and protein data and compute fractions**

In [12]:
# Compute fractions
mrna_fracs = compute_mrna_fractions(
    '../data/s.pombe/normalized_mRNA_counts.xlsx',
    '../data/s.pombe/growth_info.xlsx',
    ribo_ids, mito_ids
)

prot_fracs = compute_protein_fractions(
    '../data/s.pombe/proteomics_analysis_istvan.xlsx',
    '../data/s.pombe/growth_info.xlsx',
    ribo_ids, mito_ids
)

# Define arrays
mrna_growth  = mrna_fracs['Growth rate'].values
mrna_r_frac  = mrna_fracs['R'].values
mrna_m_frac  = mrna_fracs['M'].values
mrna_q_frac  = mrna_fracs['Q'].values

protein_growth = prot_fracs['Growth rate'].values
R_P_frac = prot_fracs['R'].values
M_P_frac = prot_fracs['M'].values
Q_P_frac = prot_fracs['Q'].values


**Fit to bayesian model and output plots with data overlayed**

In [13]:
trace  = fit_model_spombe(protein_growth, R_P_frac, M_P_frac, Q_P_frac,
                           mrna_growth, mrna_r_frac, mrna_m_frac, mrna_q_frac)
preds  = predict_from_trace_spombe(trace, protein_growth, mrna_growth)
p_prot = plot_protein_fractions_spombe(protein_growth, R_P_frac, M_P_frac, Q_P_frac, preds)
p_mrna = plot_mrna_fractions_spombe(mrna_growth, mrna_r_frac, mrna_m_frac, mrna_q_frac, preds)
az.summary(trace)


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [gamma_1, alpha_q0, sigma_d, d_mrna_c, d_mrna_q, sigma_protein, sigma_mrna, dmrna/gamma_2]


Output()

Sampling 4 chains for 8_000 tune and 6_000 draw iterations (32_000 + 24_000 draws total) took 45 seconds.
There were 19075 divergences after tuning. Increase `target_accept` or reparameterize.


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
gamma_1,3.234,0.144,2.963,3.505,0.003,0.002,2187.0,3115.0,1.0
alpha_q0,0.900,0.011,0.878,0.917,0.000,0.000,1421.0,1644.0,1.0
sigma_d,0.932,0.153,0.660,1.223,0.003,0.002,2687.0,3296.0,1.0
d_mrna_c,14.431,6.395,6.354,24.997,0.129,0.309,1887.0,2046.0,1.0
d_mrna_q,4.294,0.275,3.792,4.819,0.006,0.004,2188.0,2753.0,1.0
sigma_protein,0.012,0.002,0.009,0.015,0.000,0.000,3215.0,3900.0,1.0
sigma_mrna,0.038,0.003,0.032,0.044,0.000,0.000,3707.0,4859.0,1.0
dmrna/gamma_2,4.579,2.474,1.062,8.912,0.047,0.051,2811.0,3878.0,1.0


In [14]:
%load_ext watermark
%watermark -v -p numpy,pandas,sys,bokeh,arviz,pymc

Python implementation: CPython
Python version       : 3.12.8
IPython version      : 8.30.0

numpy : 1.26.4
pandas: 2.2.3
sys   : 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:19:53) [Clang 18.1.8 ]
bokeh : 3.7.3
arviz : 0.21.0
pymc  : 5.19.1

